<a href="https://colab.research.google.com/github/asfiya-tehmeen/Clinical-Trial-Patient-Matching-Agent/blob/main/display.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Display ranked results

def verdict_icon(v):
    return {"PASS": "✅", "FAIL": "❌", "UNCERTAIN": "⚠️ ", "MISSING_DATA": "🔬"}.get(v, "❓")

def print_match_card(result: dict, rank: int):
    score = result["match_score"]
    bar   = "█" * int(score * 10) + "░" * (10 - int(score * 10))

    print(f"\n{'='*65}")
    print(f"  MATCH #{rank}  |  Score: {score:.0%}  [{bar}]")
    print(f"{'='*65}")
    print(f"  📋 {result['trial_title']}")
    print(f"  🔑 {result.get('trial_nct_id','N/A')}  |  Phase: {result.get('trial_phase','?')}  |  Sponsor: {result.get('sponsor','?')[:38]}")
    if result.get("trial_sites"):
        print(f"  📍 Sites: {', '.join(result['trial_sites'][:3])}")
    print(f"  🔗 {result.get('trial_url','')}")

    print(f"\n  📝 Summary:")
    summary = result.get("match_summary", "")
    for line in [summary[i:i+62] for i in range(0, len(summary), 62)]:
        print(f"     {line}")

    criteria = result.get("criteria_results", [])
    if criteria:
        print(f"\n  📊 Criteria breakdown ({len(criteria)} checked):")
        for c in criteria[:8]:
            icon = verdict_icon(c["verdict"])
            text = c["criterion_text"][:55]
            print(f"     {icon}  {text}")
            if c.get("reasoning"):
                print(f"        └─ {c['reasoning'][:70]}")
        if len(criteria) > 8:
            print(f"        ... and {len(criteria)-8} more criteria checked")

    missing = result.get("missing_data_requests", [])
    if missing:
        print(f"\n  🔬 Missing data — order these to confirm eligibility:")
        for m in missing:
            print(f"     → {m}")

print("\n" + "█"*65)
print("  CLINICAL TRIAL MATCHING RESULTS  [powered by Grok-3]")
print(f"  Patient: {patient_profile['demographics']['age']}y {patient_profile['demographics']['sex']}  |  Dx: {patient_profile['primary_diagnosis']['name']}")
print("█"*65)

if not eligible:
    print("\n  ⚠️  No eligible trials found in this search batch.")
    print("  Try adjusting search terms or expanding MAX_TRIALS in Cell 3.")
else:
    for i, match in enumerate(eligible[:TOP_K], 1):
        print_match_card(match, i)

print(f"\n{'─'*65}")
print(f"  Disqualified: {len(ineligible)}  |  Eligible: {len(eligible)}")
print(f"  Showing top {min(TOP_K, len(eligible))} of {len(eligible)} eligible matches")
print(f"{'─'*65}")




█████████████████████████████████████████████████████████████████
  CLINICAL TRIAL MATCHING RESULTS  [powered by Grok-3]
  Patient: 61y F  |  Dx: Stage IIIB non-small cell lung cancer
█████████████████████████████████████████████████████████████████

  MATCH #1  |  Score: 76%  [███████░░░]
  📋 Osimertinib With or Without Bevacizumab as Initial Treatment for Patients With EGFR-Mutant Lung Cancer
  🔑 NCT04181060  |  Phase: PHASE3  |  Sponsor: National Cancer Institute (NCI)
  📍 Sites: Anchorage, United States
  🔗 https://clinicaltrials.gov/study/NCT04181060

  📝 Summary:
     The patient appears to be a good candidate for this trial, wit
     h a confirmed NSCLC diagnosis, advanced disease, and an EGFR e
     xon 19 deletion. However, additional information is needed to 
     confirm eligibility, including measurable disease status and a
     bsence of hemoptysis and significant proteinuria.

  📊 Criteria breakdown (11 checked):
     ✅  NSCLC diagnosis
        └─ Patient has a pathologi

In [ ]:
# Missing data action plan

# Tests/data that would unlock more trial matches for this patient

all_missing = {}
for result in eligible + ineligible:
    for item in result.get("missing_data_requests", []):
        if item not in all_missing:
            all_missing[item] = []
        all_missing[item].append(result.get("trial_nct_id", ""))

if all_missing:
    print("\n" + "="*65)
    print("  🔬 MISSING DATA ACTION PLAN")
    print("  Order these tests to potentially unlock more trial matches:")
    print("="*65)
    for test, trials in sorted(all_missing.items(), key=lambda x: -len(x[1])):
        print(f"\n  → {test}")
        print(f"     Relevant to: {', '.join(trials[:4])}")
    print()
else:
    print("\n✅ No critical missing data identified.")


  🔬 MISSING DATA ACTION PLAN
  Order these tests to potentially unlock more trial matches:

  → Cardiovascular evaluation
     Relevant to: NCT07003490, NCT06788912

  → RECIST v1.1 assessment
     Relevant to: NCT06734182, NCT07288034

  → Total bilirubin level
     Relevant to: NCT07288034, NCT05713994

  → Surgical evaluation
     Relevant to: NCT06788912, NCT04406831

  → Baseline measurements of sites of disease
     Relevant to: NCT04181060

  → Hemoptysis assessment
     Relevant to: NCT04181060

  → Urinalysis results
     Relevant to: NCT04181060

  → Patient's willingness to sign informed consent
     Relevant to: NCT06041776

  → Brain imaging results to confirm absence of metastasis
     Relevant to: NCT06041776

  → Postoperative wound healing status and time since surgery
     Relevant to: NCT06041776

  → Serum pregnancy test results for female patients
     Relevant to: NCT06041776

  → Patient's medical history regarding other malignancies
     Relevant to: NCT0604177

In [ ]:
# Summary table


rows = []
for r in all_results:
    criteria = r.get("criteria_results", [])
    rows.append({
        "NCT ID"    : r.get("trial_nct_id", ""),
        "Title"     : r.get("trial_title", "")[:45],
        "Phase"     : r.get("trial_phase", ""),
        "Score"     : f"{r['match_score']:.0%}",
        "PASS"      : sum(1 for c in criteria if c["verdict"] == "PASS"),
        "FAIL"      : sum(1 for c in criteria if c["verdict"] == "FAIL"),
        "UNCERTAIN" : sum(1 for c in criteria if c["verdict"] == "UNCERTAIN"),
        "MISSING"   : sum(1 for c in criteria if c["verdict"] == "MISSING_DATA"),
        "Eligible"  : "✅" if r["match_score"] > 0 else "❌"
    })

df = pd.DataFrame(rows).sort_values("Score", ascending=False)
print("\n Full results table:\n")
print(df.to_string(index=False))



 Full results table:

     NCT ID                                         Title  Phase Score  PASS  FAIL  UNCERTAIN  MISSING Eligible
NCT04181060 Osimertinib With or Without Bevacizumab as In PHASE3   76%     8     0          3        0        ✅
NCT06041776 Adjuvant Befotertinib in Stage IB-IIIB Non-sm PHASE3   37%     5     0          2        5        ✅
NCT07003490 Radical Resection With Contralateral Lymph No     NA   17%     3     0          6        0        ✅
NCT06734182 Neoadjuvant Envafolimab Plus Disitamab Vedoti PHASE2    0%     4     1          1        3        ❌
NCT04406831 The Role of MicroRNA in the Diagnosis, Progno    N/A    0%     1     4          1        1        ❌
NCT05665361 Palbociclib and Sasanlimab for the Treatment  PHASE1    0%     5     1          0        0        ❌
NCT04105270 RMT in Combination With Durvalumab + Chemo in PHASE2    0%     2     2          3        2        ❌
NCT05661786 Clinical Outcomes of HBeAg-negative CHB Patie    N/A    0%     1     